# Clase 2 — Políticas, Funciones de Valor, Ecuación de Bellman y Bandidos (versión R)

**Curso:** Aprendizaje por Refuerzo: Fundamentos y Aplicaciones
**Institución:** Universidad Austral — Facultad de Ingeniería (Posgrados)
**Docente:** Dr. Darío Ezequiel Díaz
**Fecha:** 12 de mayo de 2026

---

## Resumen del cuaderno

Esta es la contraparte en **R** del cuaderno Python `Clase02_Bellman_Bandits.ipynb`. Reproduce los nueve bloques temáticos del original idiomáticamente en R, usando:

- **`R6`** para clases orientadas a objetos (entornos y bandidos).
- **`ggplot2` + `patchwork`** para visualizaciones de calidad de publicación.
- **`reticulate`** opcionalmente, como puente con Python para corroborar resultados.

Los nueve bloques son:

1. **GridWorld** como clase R6.
2. **Resolución cerrada** de Bellman: $V^\pi = (I - \gamma P^\pi)^{-1} R^\pi$.
3. **Evaluación iterativa** con verificación de convergencia geométrica.
4. **Bandido gaussiano** de diez brazos.
5. **Estrategia $\varepsilon$-greedy**.
6. **Estrategia UCB1**.
7. **Thompson sampling** gaussiano.
8. **Comparación experimental** sobre múltiples réplicas.
9. **Análisis estadístico** con `wilcox.test` y `t.test`.

> **Compatibilidad:** se ejecuta en R 4.3 o superior, tanto local como en Google Colab con el kernel `ir` instalado. La sección 4.1 puede usar `reticulate` para comparar contra NumPy si el entorno conda `rl-docencia` está disponible; en su ausencia, todo el cuaderno corre puramente en R.

## Bloque 0 — Configuración del entorno

Cargamos paquetes mínimos siguiendo el patrón estándar del curso. Si el script personal de setup está disponible en Drive, se prioriza; de lo contrario, se instalan los paquetes esenciales sobre la marcha.

In [ ]:
# --- Paso 1: paquetes R ---
# Si dispone de su script de setup personal en Drive, sourcearlo.
# En caso contrario, instalar los paquetes mínimos necesarios para esta clase.
ruta_setup_personal <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"
if (file.exists(ruta_setup_personal)) {
  cat("Cargando setup personal desde Drive...\n")
  source(ruta_setup_personal)
} else {
  cat("Setup personal no encontrado. Instalando paquetes mínimos para Clase 2...\n")
  paquetes_clase2 <- c("R6", "ggplot2", "patchwork", "reticulate")
  faltantes <- paquetes_clase2[!sapply(paquetes_clase2,
                                        function(p) requireNamespace(p, quietly = TRUE))]
  if (length(faltantes) > 0) {
    install.packages(faltantes, quiet = TRUE)
  }
  cat("Paquetes verificados.\n")
}
library(R6)
library(ggplot2)
library(patchwork)
# reticulate se carga sólo si se desea comparar con Python; ver sección opcional.

# Semilla global para reproducibilidad
SEMILLA_GLOBAL <- 42L
set.seed(SEMILLA_GLOBAL)

# Paleta institucional (consistente con la presentación y el cuaderno Python)
COLOR_NAVY        <- "#1E3A5F"
COLOR_ORANGE      <- "#D86A2C"
COLOR_TEAL        <- "#2E8A99"
COLOR_LIGHT_TEAL  <- "#9FD3D8"
COLOR_LIGHT_ORANGE<- "#F2A06B"
COLOR_GRAY        <- "#4A4A4A"

# Tema gráfico institucional
tema_austral <- theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(color = COLOR_NAVY, face = "bold", size = 12),
    axis.title = element_text(color = COLOR_NAVY),
    axis.text  = element_text(color = COLOR_GRAY),
    panel.grid.minor = element_blank(),
    legend.position = "right"
  )
theme_set(tema_austral)

cat("Entorno configurado correctamente.\n")
cat("R version:", R.version.string, "\n")

## Bloque 1 — GridWorld como clase R6

Definimos el entorno mediante el sistema de clases R6, paradigma orientado a objetos con semántica de referencia (en contraste con S4, que usa copias). R6 es la elección idiomática para entornos de RL en R porque permite mantener estado mutable de manera eficiente.

**Convenciones:**

- Coordenadas $(r, c)$ con fila cero arriba.
- Estado inicial $S = (3, 0)$, meta $G = (0, 3)$.
- Muros en $(2, 1)$ y $(2, 2)$.
- Acciones $0$ a $3$: arriba, derecha, abajo, izquierda.
- Recompensa de paso: $-0.1$. Recompensa terminal: $+1$.

In [ ]:
# Definición de la clase GridWorld con R6
GridWorld <- R6Class("GridWorld",
  public = list(
    n_filas = NULL,
    n_cols = NULL,
    muros = NULL,
    inicio = NULL,
    meta = NULL,
    recompensa_paso = NULL,
    recompensa_meta = NULL,
    acciones = NULL,
    nombres_acciones = NULL,
    estados = NULL,
    idx_estado = NULL,
    n_estados = NULL,
    n_acciones = NULL,

    initialize = function(n_filas = 4, n_cols = 4,
                          muros = list(),
                          inicio = c(3, 0), meta = c(0, 3),
                          recompensa_paso = -0.1,
                          recompensa_meta = 1.0) {
      self$n_filas <- n_filas
      self$n_cols <- n_cols
      self$muros <- muros
      self$inicio <- inicio
      self$meta <- meta
      self$recompensa_paso <- recompensa_paso
      self$recompensa_meta <- recompensa_meta

      # Acciones codificadas como desplazamientos (delta_fila, delta_col)
      self$acciones <- list(c(-1, 0), c(0, 1), c(1, 0), c(0, -1))
      self$nombres_acciones <- c("\u2191", "\u2192", "\u2193", "\u2190")

      # Construir lista de estados válidos
      self$estados <- list()
      for (r in 0:(n_filas - 1)) {
        for (c in 0:(n_cols - 1)) {
          s <- c(r, c)
          if (!private$.es_muro(s)) {
            self$estados <- append(self$estados, list(s))
          }
        }
      }
      self$n_estados <- length(self$estados)
      self$n_acciones <- length(self$acciones)

      # Crear índice de estado: clave "r,c" -> entero
      claves <- sapply(self$estados, function(s) paste(s, collapse = ","))
      self$idx_estado <- setNames(seq_len(self$n_estados), claves)
    },

    # Aplica una acción y devuelve lista con estado siguiente, recompensa y terminal
    transicion = function(estado, accion) {
      # Estado terminal absorbente
      if (estado[1] == self$meta[1] && estado[2] == self$meta[2]) {
        return(list(siguiente = estado, recompensa = 0, terminal = TRUE))
      }

      delta <- self$acciones[[accion]]
      nr <- estado[1] + delta[1]
      nc <- estado[2] + delta[2]
      nuevo <- c(nr, nc)

      fuera <- (nr < 0 || nr >= self$n_filas || nc < 0 || nc >= self$n_cols)
      siguiente <- if (fuera || private$.es_muro(nuevo)) estado else nuevo

      if (siguiente[1] == self$meta[1] && siguiente[2] == self$meta[2]) {
        return(list(siguiente = siguiente, recompensa = self$recompensa_meta, terminal = TRUE))
      }
      list(siguiente = siguiente, recompensa = self$recompensa_paso, terminal = FALSE)
    },

    # Devuelve el índice (entero) de un estado dado por coordenadas
    indice = function(s) {
      clave <- paste(s, collapse = ",")
      self$idx_estado[[clave]]
    }
  ),

  private = list(
    .es_muro = function(s) {
      if (length(self$muros) == 0) return(FALSE)
      for (m in self$muros) {
        if (s[1] == m[1] && s[2] == m[2]) return(TRUE)
      }
      FALSE
    }
  )
)

# Instanciar el GridWorld con los muros de la presentación
gw <- GridWorld$new(muros = list(c(2, 1), c(2, 2)))

cat(sprintf("Espacio de estados: %d estados válidos\n", gw$n_estados))
cat(sprintf("Espacio de acciones: %d acciones\n", gw$n_acciones))
cat(sprintf("Inicio: (%d, %d), Meta: (%d, %d)\n",
            gw$inicio[1], gw$inicio[2], gw$meta[1], gw$meta[2]))
cat("Muros:", paste(sapply(gw$muros,
                           function(m) sprintf("(%d,%d)", m[1], m[2])),
                    collapse = ", "), "\n")

### Visualización del entorno con ggplot2

Construimos una función de dibujo basada en `ggplot2` que recibe el entorno, opcionalmente un vector de valores, y produce una grilla con celdas coloreadas según su rol. La invertimos verticalmente para que coincida con la convención de la presentación: $S$ abajo-izquierda y $G$ arriba-derecha.

In [ ]:
dibujar_gridworld <- function(env, valores = NULL, politica = NULL,
                              titulo = "GridWorld",
                              mostrar_acciones = FALSE) {
  # Construir data frame con todas las celdas y sus atributos
  filas <- 0:(env$n_filas - 1)
  cols <- 0:(env$n_cols - 1)
  cuadricula <- expand.grid(fila = filas, col = cols)

  cuadricula$tipo <- "vacia"
  cuadricula$y_plot <- env$n_filas - 1 - cuadricula$fila   # invertir verticalmente
  cuadricula$etiqueta <- ""

  # Asignar tipo a cada celda
  for (i in seq_len(nrow(cuadricula))) {
    s <- c(cuadricula$fila[i], cuadricula$col[i])
    if (any(sapply(env$muros, function(m) all(s == m)))) {
      cuadricula$tipo[i] <- "muro"
    } else if (all(s == env$inicio)) {
      cuadricula$tipo[i] <- "inicio"
      cuadricula$etiqueta[i] <- "S"
    } else if (all(s == env$meta)) {
      cuadricula$tipo[i] <- "meta"
      cuadricula$etiqueta[i] <- "G"
    }
  }

  # Si hay valores, asignarlos como etiqueta numérica
  if (!is.null(valores)) {
    for (i in seq_len(nrow(cuadricula))) {
      s <- c(cuadricula$fila[i], cuadricula$col[i])
      if (cuadricula$tipo[i] %in% c("vacia")) {
        idx <- env$indice(s)
        if (!is.null(idx)) {
          cuadricula$etiqueta[i] <- sprintf("%.2f", valores[idx])
        }
      }
    }
  }

  paleta <- c(vacia = "white", muro = COLOR_GRAY,
              inicio = COLOR_LIGHT_TEAL, meta = COLOR_LIGHT_ORANGE)

  p <- ggplot(cuadricula, aes(x = col, y = y_plot, fill = tipo)) +
    geom_tile(color = COLOR_NAVY, linewidth = 0.6) +
    geom_text(aes(label = etiqueta), color = COLOR_NAVY,
              fontface = "bold", size = 3.8) +
    scale_fill_manual(values = paleta, guide = "none") +
    coord_fixed() +
    scale_x_continuous(breaks = NULL, expand = c(0, 0)) +
    scale_y_continuous(breaks = NULL, expand = c(0, 0)) +
    labs(title = titulo, x = NULL, y = NULL) +
    theme(panel.grid = element_blank(),
          axis.text = element_blank(),
          axis.ticks = element_blank(),
          panel.border = element_rect(color = COLOR_NAVY, fill = NA, linewidth = 0.8))

  p
}

options(repr.plot.width = 5, repr.plot.height = 5)
print(dibujar_gridworld(gw, titulo = "GridWorld 4x4: S inicial, G meta, muros centrales"))

## Bloque 2 — Resolución cerrada de la ecuación de Bellman

Reproducimos en R la fórmula matricial:

$$
V^\pi = (I - \gamma P^\pi)^{-1} R^\pi.
$$

La rutina `construir_dinamica_inducida` marginaliza sobre acciones para obtener la matriz de transición y el vector de recompensa bajo la política dada. Después, `evaluar_politica_exacta` resuelve el sistema lineal usando `solve()`, que internamente emplea descomposición LU.

In [ ]:
construir_dinamica_inducida <- function(env, politica) {
  n <- env$n_estados
  P_pi <- matrix(0, nrow = n, ncol = n)
  R_pi <- numeric(n)

  for (i in seq_along(env$estados)) {
    s <- env$estados[[i]]
    for (a in seq_len(env$n_acciones)) {
      res <- env$transicion(s, a)
      j <- env$indice(res$siguiente)
      P_pi[i, j] <- P_pi[i, j] + politica[i, a]
      R_pi[i] <- R_pi[i] + politica[i, a] * res$recompensa
    }
  }
  list(P = P_pi, R = R_pi)
}

# Política equiprobable
pol_uniforme <- matrix(1 / gw$n_acciones,
                       nrow = gw$n_estados, ncol = gw$n_acciones)

din <- construir_dinamica_inducida(gw, pol_uniforme)
P_pi <- din$P
R_pi <- din$R

cat("Verificación: las filas de P^pi deben sumar 1\n")
cat(sprintf("  min suma fila = %.6f\n", min(rowSums(P_pi))))
cat(sprintf("  max suma fila = %.6f\n", max(rowSums(P_pi))))

gamma_dsc <- 0.95
cat(sprintf("\nNorma infinito de gamma * P^pi = %.4f (debe ser < 1)\n",
            gamma_dsc * max(rowSums(abs(P_pi)))))

In [ ]:
evaluar_politica_exacta <- function(env, politica, gamma) {
  din <- construir_dinamica_inducida(env, politica)
  n <- env$n_estados
  A <- diag(n) - gamma * din$P
  V <- solve(A, din$R)
  V
}

V_exacta <- evaluar_politica_exacta(gw, pol_uniforme, gamma_dsc)

# Tabla de valores por estado
cat("Valores V^pi por inversión exacta (gamma = 0.95):\n\n")
cat(sprintf("%-10s %10s\n", "Estado", "V^pi(s)"))
cat(paste(rep("-", 22), collapse = ""), "\n")
for (i in seq_along(gw$estados)) {
  s <- gw$estados[[i]]
  cat(sprintf("(%d,%d)      %10.4f\n", s[1], s[2], V_exacta[i]))
}

### Visualización del mapa de valores

Reutilizamos `dibujar_gridworld` pasándole el vector $V^\pi$.

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5)
p_val <- dibujar_gridworld(gw, valores = V_exacta,
                            titulo = expression(paste(V^pi,
                              " bajo política equiprobable (",
                              gamma, " = 0.95)")))
print(p_val)

cat("\nInterpretación: los valores son negativos porque la política aleatoria\n",
    "acumula múltiples penalizaciones de -0.1 antes de alcanzar la meta. Los\n",
    "valores crecen monótonamente hacia G, mostrando que la cercanía a la\n",
    "meta es ventajosa incluso bajo aleatoriedad.\n", sep="")

## Bloque 3 — Evaluación iterativa y convergencia geométrica

Implementamos la iteración del operador de Bellman $V_{k+1} = \mathcal{T}^\pi V_k$ y rastreamos el incremento $\Delta_k = \|V_{k+1} - V_k\|_\infty$ en cada paso. Después comparamos la pendiente empírica del decaimiento logarítmico con la cota teórica $\log \gamma$.

In [ ]:
evaluacion_iterativa <- function(env, politica, gamma,
                                  theta = 1e-10, max_iter = 10000) {
  din <- construir_dinamica_inducida(env, politica)
  P_pi <- din$P
  R_pi <- din$R

  V <- numeric(env$n_estados)
  historia_delta <- numeric(0)
  V_iteraciones <- list(V)

  for (k in seq_len(max_iter)) {
    V_nuevo <- as.numeric(R_pi + gamma * P_pi %*% V)
    delta <- max(abs(V_nuevo - V))
    historia_delta <- c(historia_delta, delta)
    V_iteraciones <- append(V_iteraciones, list(V_nuevo))
    V <- V_nuevo
    if (delta < theta) break
  }

  list(V = V, historia_delta = historia_delta,
       V_iteraciones = V_iteraciones, n_iter = k)
}

res_iter <- evaluacion_iterativa(gw, pol_uniforme, gamma_dsc, theta = 1e-10)

cat(sprintf("Iteraciones hasta convergencia (tolerancia 1e-10): %d\n",
            res_iter$n_iter))
cat(sprintf("Error final respecto de la solución exacta: %.2e\n",
            max(abs(res_iter$V - V_exacta))))
cat(sprintf("\nPrimeros 5 deltas: %s\n",
            paste(sprintf("%.3e", head(res_iter$historia_delta, 5)),
                  collapse = ", ")))
cat(sprintf("Últimos 5 deltas: %s\n",
            paste(sprintf("%.3e", tail(res_iter$historia_delta, 5)),
                  collapse = ", ")))

### Verificación empírica de la convergencia geométrica

Si la cota teórica $\|V_k - V^\pi\|_\infty \leq \gamma^k \|V_0 - V^\pi\|_\infty$ se manifestara con igualdad, el logaritmo del incremento decrecería linealmente con pendiente $\log \gamma$. Empíricamente la convergencia suele ser más rápida porque la tasa exacta queda determinada por el radio espectral de $\gamma P^\pi$, no por su norma infinito.

In [ ]:
n_ajuste <- min(200, length(res_iter$historia_delta))
deltas <- res_iter$historia_delta[1:n_ajuste]
ks <- 0:(n_ajuste - 1)

ajuste <- lm(log(deltas) ~ ks)
pendiente <- coef(ajuste)[2]

cat(sprintf("Pendiente empírica del decaimiento logarítmico: %.4f\n",
            pendiente))
cat(sprintf("log(gamma) teórico: %.4f\n", log(gamma_dsc)))
cat(sprintf("Cociente empírico/teórico: %.4f\n",
            pendiente / log(gamma_dsc)))

# Construir data frame para graficar
n_total <- length(res_iter$historia_delta)
df_conv <- data.frame(
  k = 0:(n_total - 1),
  empirico = res_iter$historia_delta,
  teorico = res_iter$historia_delta[1] * gamma_dsc^(0:(n_total - 1))
)

options(repr.plot.width = 8, repr.plot.height = 4.5)
ggplot(df_conv, aes(x = k)) +
  geom_line(aes(y = empirico, color = "Empirico"), linewidth = 1.2) +
  geom_line(aes(y = teorico, color = "Teorico"),
            linewidth = 1.0, linetype = "dashed") +
  scale_y_log10() +
  scale_color_manual(
    name = NULL,
    values = c("Empirico" = COLOR_NAVY, "Teorico" = COLOR_ORANGE),
    labels = c(expression(Delta[k]~empirico),
               expression(Delta[0]%.%gamma^k~teorico))
  ) +
  labs(title = "Convergencia geométrica del operador de Bellman",
       x = expression("Iteración "*k),
       y = expression("||"*V[k+1]-V[k]*"||"[infinity]*"  (log)")) +
  theme(legend.position = c(0.85, 0.85),
        legend.background = element_rect(fill = "white", color = COLOR_NAVY))

## Bloque 4 — Bandido gaussiano de diez brazos

Definimos la clase `BanditoGaussiano` mediante R6. Cada instancia tiene un vector de medias verdaderas $q_*(a)$ muestreadas de $\mathcal{N}(0, 1)$, fijas durante toda la vida del bandido. Las recompensas observadas son $\mathcal{N}(q_*(a), \sigma_r^2)$.

In [ ]:
BanditoGaussiano <- R6Class("BanditoGaussiano",
  public = list(
    k = NULL,
    sigma_r = NULL,
    q_estrella = NULL,
    brazo_optimo = NULL,

    initialize = function(k = 10, sigma_r = 1.0, semilla = NULL) {
      self$k <- k
      self$sigma_r <- sigma_r
      if (!is.null(semilla)) set.seed(semilla)
      self$q_estrella <- rnorm(k, mean = 0, sd = 1)
      self$brazo_optimo <- which.max(self$q_estrella)
    },

    jugar = function(accion) {
      rnorm(1, mean = self$q_estrella[accion], sd = self$sigma_r)
    }
  )
)

# Bandido demostrativo con semilla 2024
bandido_demo <- BanditoGaussiano$new(k = 10, semilla = 2024)

cat("Medias verdaderas q*(a) del bandido demostrativo:\n\n")
cat(sprintf("%-8s %10s %10s\n", "Brazo", "q*(a)", "Optimo"))
cat(paste(rep("-", 30), collapse = ""), "\n")
for (a in 1:bandido_demo$k) {
  marca <- if (a == bandido_demo$brazo_optimo) "*" else " "
  cat(sprintf("%-8d %+10.3f %10s\n", a, bandido_demo$q_estrella[a], marca))
}

### Visualización al estilo Sutton-Barto figura 2.2

Muestreamos dos mil recompensas independientes de cada brazo y graficamos violines con `ggplot2`. La línea horizontal naranja en cada violín marca la media verdadera $q_*(a)$.

In [ ]:
set.seed(7)
n_muestras <- 2000

# Construir data frame en formato largo: una fila por (brazo, muestra)
df_band <- do.call(rbind, lapply(1:bandido_demo$k, function(a) {
  data.frame(
    brazo = factor(a, levels = 1:bandido_demo$k),
    recompensa = rnorm(n_muestras,
                       bandido_demo$q_estrella[a],
                       bandido_demo$sigma_r)
  )
}))

# Data frame con las medias verdaderas
df_medias <- data.frame(
  brazo = factor(1:bandido_demo$k, levels = 1:bandido_demo$k),
  q_estrella = bandido_demo$q_estrella
)

options(repr.plot.width = 9, repr.plot.height = 5)
ggplot(df_band, aes(x = brazo, y = recompensa)) +
  geom_violin(fill = COLOR_LIGHT_TEAL, color = COLOR_NAVY, alpha = 0.7) +
  geom_segment(data = df_medias,
               aes(x = as.numeric(brazo) - 0.35,
                   xend = as.numeric(brazo) + 0.35,
                   y = q_estrella, yend = q_estrella),
               color = COLOR_ORANGE, linewidth = 1.2) +
  geom_hline(yintercept = 0, color = "gray60",
             linewidth = 0.3, alpha = 0.5) +
  labs(title = sprintf("Distribución de recompensas del bandido gaussiano (k=10, óptimo: a=%d)",
                       bandido_demo$brazo_optimo),
       x = "Brazo a",
       y = "Recompensa R")

## Bloque 5 — Estrategia $\varepsilon$-greedy

Con probabilidad $1 - \varepsilon$ se elige el brazo con mayor estimación actual; con probabilidad $\varepsilon$ se elige uniformemente al azar. La actualización del promedio empírico se hace incrementalmente:

$$
\hat{Q}_{n+1}(a) = \hat{Q}_n(a) + \frac{1}{n+1}[R_{n+1} - \hat{Q}_n(a)].
$$

In [ ]:
epsilon_greedy <- function(bandido, T_h, epsilon, semilla = NULL) {
  if (!is.null(semilla)) set.seed(semilla)
  k <- bandido$k
  Q_hat <- numeric(k)
  N <- integer(k)
  recompensas <- numeric(T_h)
  acciones_opt <- logical(T_h)

  for (t in seq_len(T_h)) {
    if (runif(1) < epsilon) {
      a <- sample.int(k, 1)   # exploración uniforme
    } else {
      mx <- max(Q_hat)
      empates <- which(Q_hat == mx)
      a <- if (length(empates) == 1) empates else sample(empates, 1)
    }
    r <- bandido$jugar(a)
    N[a] <- N[a] + 1
    Q_hat[a] <- Q_hat[a] + (r - Q_hat[a]) / N[a]
    recompensas[t] <- r
    acciones_opt[t] <- (a == bandido$brazo_optimo)
  }
  list(recompensas = recompensas, acciones_opt = acciones_opt)
}

res_eg <- epsilon_greedy(bandido_demo, T_h = 1000,
                         epsilon = 0.1, semilla = 1)
cat(sprintf("eps-greedy (eps=0.1) en 1000 pasos:\n"))
cat(sprintf("  recompensa media últimos 200 = %.3f\n",
            mean(tail(res_eg$recompensas, 200))))
cat(sprintf("  fracción óptima últimos 200 = %.3f\n",
            mean(tail(res_eg$acciones_opt, 200))))

## Bloque 6 — Estrategia UCB1

Optimismo bajo incertidumbre: en cada paso elegimos

$$
A_t = \arg\max_a \left[\hat{Q}_t(a) + c\sqrt{\frac{\ln t}{N_t(a)}}\right].
$$

La fase inicial juega cada brazo una vez para evitar la división por cero.

In [ ]:
ucb1 <- function(bandido, T_h, c = 2.0, semilla = NULL) {
  if (!is.null(semilla)) set.seed(semilla)
  k <- bandido$k
  Q_hat <- numeric(k)
  N <- integer(k)
  recompensas <- numeric(T_h)
  acciones_opt <- logical(T_h)

  # Fase inicial: jugar cada brazo una vez
  for (a in seq_len(min(k, T_h))) {
    r <- bandido$jugar(a)
    N[a] <- 1
    Q_hat[a] <- r
    recompensas[a] <- r
    acciones_opt[a] <- (a == bandido$brazo_optimo)
  }

  # Bucle principal
  if (T_h > k) {
    for (t in (k + 1):T_h) {
      bono <- c * sqrt(log(t) / N)
      a <- which.max(Q_hat + bono)
      r <- bandido$jugar(a)
      N[a] <- N[a] + 1
      Q_hat[a] <- Q_hat[a] + (r - Q_hat[a]) / N[a]
      recompensas[t] <- r
      acciones_opt[t] <- (a == bandido$brazo_optimo)
    }
  }
  list(recompensas = recompensas, acciones_opt = acciones_opt)
}

res_ucb <- ucb1(bandido_demo, T_h = 1000, c = 2.0, semilla = 2)
cat(sprintf("UCB1 (c=2.0) en 1000 pasos:\n"))
cat(sprintf("  recompensa media últimos 200 = %.3f\n",
            mean(tail(res_ucb$recompensas, 200))))
cat(sprintf("  fracción óptima últimos 200 = %.3f\n",
            mean(tail(res_ucb$acciones_opt, 200))))

## Bloque 7 — Thompson sampling gaussiano

Para recompensas $\mathcal{N}(q_*(a), \sigma_r^2)$ con varianza conocida, la prior conjugada es una normal sobre la media. Las fórmulas de actualización tras $N$ observaciones del brazo $a$ con suma $\sum R_i$ son:

$$
\sigma_{\text{post}}^2 = \left(\frac{1}{\sigma_0^2} + \frac{N}{\sigma_r^2}\right)^{-1},
\qquad
\mu_{\text{post}} = \sigma_{\text{post}}^2\left(\frac{\mu_0}{\sigma_0^2} + \frac{\sum R_i}{\sigma_r^2}\right).
$$

In [ ]:
thompson_gaussiano <- function(bandido, T_h,
                                mu_prior = 0.0, var_prior = 1.0,
                                semilla = NULL) {
  if (!is.null(semilla)) set.seed(semilla)
  k <- bandido$k
  sigma2_r <- bandido$sigma_r^2

  mu_post <- rep(mu_prior, k)
  var_post <- rep(var_prior, k)
  sum_r <- numeric(k)
  N <- integer(k)

  recompensas <- numeric(T_h)
  acciones_opt <- logical(T_h)

  for (t in seq_len(T_h)) {
    # Muestreo de la posterior de cada brazo
    muestras <- rnorm(k, mu_post, sqrt(var_post))
    a <- which.max(muestras)
    r <- bandido$jugar(a)
    N[a] <- N[a] + 1
    sum_r[a] <- sum_r[a] + r
    # Actualización conjugada
    var_post[a] <- 1 / (1 / var_prior + N[a] / sigma2_r)
    mu_post[a] <- var_post[a] * (mu_prior / var_prior + sum_r[a] / sigma2_r)
    recompensas[t] <- r
    acciones_opt[t] <- (a == bandido$brazo_optimo)
  }
  list(recompensas = recompensas, acciones_opt = acciones_opt)
}

res_ts <- thompson_gaussiano(bandido_demo, T_h = 1000, semilla = 3)
cat(sprintf("Thompson sampling en 1000 pasos:\n"))
cat(sprintf("  recompensa media últimos 200 = %.3f\n",
            mean(tail(res_ts$recompensas, 200))))
cat(sprintf("  fracción óptima últimos 200 = %.3f\n",
            mean(tail(res_ts$acciones_opt, 200))))

## Bloque 8 — Comparación experimental sobre múltiples réplicas

Ejecutamos cada estrategia sobre 300 réplicas independientes con bandidos distintos. Esto requiere unos segundos. Para acelerar, podríamos paralelizar con `parallel::mclapply`, pero por simplicidad mantenemos la versión secuencial.

In [ ]:
ejecutar_replicas <- function(algoritmo, n_replicas, T_h, args_alg,
                              semilla_base = 10000L, k = 10) {
  regret_mat <- matrix(0, nrow = n_replicas, ncol = T_h)
  opt_mat <- matrix(FALSE, nrow = n_replicas, ncol = T_h)

  for (i in seq_len(n_replicas)) {
    bandido_i <- BanditoGaussiano$new(k = k, semilla = semilla_base + i)
    q_opt <- bandido_i$q_estrella[bandido_i$brazo_optimo]

    args_completos <- c(list(bandido = bandido_i, T_h = T_h,
                              semilla = semilla_base + i + 99999L),
                        args_alg)
    res <- do.call(algoritmo, args_completos)
    regret_mat[i, ] <- q_opt - res$recompensas
    opt_mat[i, ] <- res$acciones_opt
  }
  list(regret = regret_mat, opt = opt_mat)
}

T_h <- 1000L
N_REP <- 300L

cat(sprintf("Ejecutando %d réplicas con horizonte T = %d para cada algoritmo...\n",
            N_REP, T_h))
cat("Esto puede tomar unos segundos.\n\n")

t0 <- Sys.time()
rep_eg01 <- ejecutar_replicas(epsilon_greedy, N_REP, T_h,
                               list(epsilon = 0.01))
cat("  eps-greedy(0.01) listo\n")
rep_eg10 <- ejecutar_replicas(epsilon_greedy, N_REP, T_h,
                               list(epsilon = 0.10))
cat("  eps-greedy(0.10) listo\n")
rep_ucb <- ejecutar_replicas(ucb1, N_REP, T_h, list(c = 2.0))
cat("  UCB1(c=2.0) listo\n")
rep_ts <- ejecutar_replicas(thompson_gaussiano, N_REP, T_h, list())
cat("  Thompson sampling listo\n")
t1 <- Sys.time()
cat(sprintf("\nTiempo total de simulación: %s\n", format(t1 - t0)))

# Curvas promedio
regret_eg01 <- colMeans(t(apply(rep_eg01$regret, 1, cumsum)))
regret_eg10 <- colMeans(t(apply(rep_eg10$regret, 1, cumsum)))
regret_ucb <- colMeans(t(apply(rep_ucb$regret, 1, cumsum)))
regret_ts <- colMeans(t(apply(rep_ts$regret, 1, cumsum)))

pct_eg01 <- colMeans(rep_eg01$opt)
pct_eg10 <- colMeans(rep_eg10$opt)
pct_ucb <- colMeans(rep_ucb$opt)
pct_ts <- colMeans(rep_ts$opt)

### 8.1. Curvas de regret y fracción óptima

Construimos un panel doble con `patchwork`: a la izquierda, el regret acumulado promedio; a la derecha, la fracción de acción óptima en cada paso.

In [ ]:
# Data frame en formato largo para ggplot2
df_curvas <- rbind(
  data.frame(t = 1:T_h, regret = regret_eg01, fraccion = pct_eg01,
             algoritmo = "eps-greedy (0.01)"),
  data.frame(t = 1:T_h, regret = regret_eg10, fraccion = pct_eg10,
             algoritmo = "eps-greedy (0.10)"),
  data.frame(t = 1:T_h, regret = regret_ucb, fraccion = pct_ucb,
             algoritmo = "UCB1 (c=2)"),
  data.frame(t = 1:T_h, regret = regret_ts, fraccion = pct_ts,
             algoritmo = "Thompson sampling")
)
df_curvas$algoritmo <- factor(df_curvas$algoritmo,
  levels = c("eps-greedy (0.01)", "eps-greedy (0.10)",
             "UCB1 (c=2)", "Thompson sampling"))

paleta_alg <- c("eps-greedy (0.01)" = COLOR_LIGHT_ORANGE,
                "eps-greedy (0.10)" = COLOR_ORANGE,
                "UCB1 (c=2)" = COLOR_NAVY,
                "Thompson sampling" = COLOR_TEAL)

p_regret <- ggplot(df_curvas, aes(x = t, y = regret, color = algoritmo)) +
  geom_line(linewidth = 1.0) +
  scale_color_manual(values = paleta_alg, name = NULL) +
  labs(title = sprintf("Regret acumulado (%d réplicas, T = %d)", N_REP, T_h),
       x = expression("Iteración "*t),
       y = expression("Regret acumulado promedio "*R[t])) +
  theme(legend.position = "bottom")

p_optimo <- ggplot(df_curvas, aes(x = t, y = fraccion, color = algoritmo)) +
  geom_line(linewidth = 1.0) +
  geom_hline(yintercept = 1, color = "gray60",
             linewidth = 0.3, alpha = 0.5) +
  scale_color_manual(values = paleta_alg, name = NULL, guide = "none") +
  ylim(0, 1.05) +
  labs(title = "Fracción de acciones óptimas",
       x = expression("Iteración "*t),
       y = "Probabilidad de elegir el brazo óptimo")

options(repr.plot.width = 14, repr.plot.height = 5)
p_regret + p_optimo + plot_layout(widths = c(1, 1))

## Bloque 9 — Análisis estadístico del regret

Cuantificamos la incertidumbre y testeamos significación con herramientas estadísticas estándar:

1. Intervalos de confianza al 95% por el método de la t de Student.
2. Pruebas pareadas de Wilcoxon, robustas a desviaciones de la normalidad.
3. Histograma comparativo de las distribuciones del regret final.

In [ ]:
# Regret final por réplica para cada algoritmo
regret_final <- list(
  "eps-greedy(0.01)" = rowSums(rep_eg01$regret),
  "eps-greedy(0.10)" = rowSums(rep_eg10$regret),
  "UCB1(c=2.0)"      = rowSums(rep_ucb$regret),
  "Thompson"          = rowSums(rep_ts$regret)
)

intervalo_confianza_95 <- function(x) {
  n <- length(x)
  m <- mean(x)
  sem <- sd(x) / sqrt(n)
  t_crit <- qt(0.975, df = n - 1)
  c(media = m, inf = m - t_crit * sem, sup = m + t_crit * sem)
}

cat(sprintf("Regret final acumulado en T = %d (intervalos de confianza al 95%%):\n\n", T_h))
cat(sprintf("%-22s %10s %12s %12s\n", "Algoritmo", "Media", "IC95% inf", "IC95% sup"))
cat(paste(rep("-", 58), collapse = ""), "\n")
for (nombre in names(regret_final)) {
  ic <- intervalo_confianza_95(regret_final[[nombre]])
  cat(sprintf("%-22s %10.2f %12.2f %12.2f\n", nombre, ic[1], ic[2], ic[3]))
}

cat("\n", paste(rep("=", 58), collapse = ""), "\n", sep="")
cat("Pruebas de Wilcoxon pareadas (regret final por réplica):\n\n")

w1 <- wilcox.test(regret_final[["Thompson"]],
                   regret_final[["UCB1(c=2.0)"]], paired = TRUE)
cat(sprintf("Thompson vs UCB1:           V = %10.1f,  p = %.4e\n",
            w1$statistic, w1$p.value))

w2 <- wilcox.test(regret_final[["UCB1(c=2.0)"]],
                   regret_final[["eps-greedy(0.10)"]], paired = TRUE)
cat(sprintf("UCB1 vs eps-greedy(0.10):   V = %10.1f,  p = %.4e\n",
            w2$statistic, w2$p.value))

w3 <- wilcox.test(regret_final[["eps-greedy(0.10)"]],
                   regret_final[["eps-greedy(0.01)"]], paired = TRUE)
cat(sprintf("eps(0.10) vs eps(0.01):     V = %10.1f,  p = %.4e\n",
            w3$statistic, w3$p.value))

cat("\nInterpretación: los p-valores extremadamente pequeños indican que las\n",
    "diferencias observadas no son explicables por azar muestral. Bajo el nivel\n",
    "convencional alpha=0.05, las ordenaciones son estadísticamente significativas.\n",
    sep="")

### 9.1. Histogramas del regret final

Los histogramas revelan información que las medias por sí solas ocultan: dispersión, asimetría y presencia de colas pesadas.

In [ ]:
df_hist <- data.frame(
  regret = c(regret_final[["eps-greedy(0.01)"]],
             regret_final[["eps-greedy(0.10)"]],
             regret_final[["UCB1(c=2.0)"]],
             regret_final[["Thompson"]]),
  algoritmo = factor(rep(c("eps-greedy (0.01)", "eps-greedy (0.10)",
                            "UCB1 (c=2)", "Thompson sampling"),
                          each = N_REP),
                     levels = c("eps-greedy (0.01)", "eps-greedy (0.10)",
                                "UCB1 (c=2)", "Thompson sampling"))
)

options(repr.plot.width = 10, repr.plot.height = 5)
ggplot(df_hist, aes(x = regret, fill = algoritmo)) +
  geom_histogram(bins = 40, alpha = 0.55, position = "identity") +
  scale_fill_manual(values = paleta_alg, name = NULL) +
  labs(title = sprintf("Distribución del regret final (%d réplicas)", N_REP),
       x = expression("Regret acumulado final "*R[T]),
       y = "Frecuencia")

## Sección opcional — Verificación cruzada con Python vía `reticulate`

Si se dispone del entorno conda `rl-docencia` con NumPy y SciPy instalados (configurado durante el curso), se puede invocar Python desde R para reproducir cálculos clave y confirmar la concordancia entre implementaciones. Esta sección es **opcional**: si el entorno conda no está disponible, basta con omitirla.

In [ ]:
# Comparación cruzada con Python (opcional)
# Esta celda solo funciona si el entorno conda 'rl-docencia' está instalado.
# En Google Colab puede que no esté disponible; en su instalación local sí.

ejecutar_comparacion <- tryCatch({
  Sys.setenv(PYTHONNOUSERSITE = "1")
  library(reticulate)
  use_condaenv("rl-docencia", required = TRUE)
  np <- import("numpy")
  TRUE
}, error = function(e) {
  cat("Entorno conda 'rl-docencia' no disponible:", conditionMessage(e), "\n")
  cat("Saltando esta sección opcional.\n")
  FALSE
})

if (ejecutar_comparacion) {
  # Construir P^pi y R^pi en R, traducirlos a NumPy, y resolver
  # el sistema lineal con np.linalg.solve para verificar concordancia.
  P_np <- np$array(P_pi)
  R_np <- np$array(R_pi)
  I_np <- np$eye(as.integer(gw$n_estados))

  A_np <- np$subtract(I_np, np$multiply(gamma_dsc, P_np))
  V_python <- np$linalg$solve(A_np, R_np)

  cat("Diferencia máxima entre V calculado en R y en Python:\n")
  cat(sprintf("  max|V_R - V_python| = %.2e\n",
              max(abs(V_exacta - as.numeric(V_python)))))
  cat("(Esperado: < 1e-12, dentro del error de redondeo de punto flotante.)\n")
}

## Ejercicios propuestos

Los siguientes ejercicios consolidan los conceptos del cuaderno y constituyen la base de la entrega evaluativa del 18 de mayo.

### Ejercicio 1 (teórico-computacional)

Demuestre numéricamente que la matriz $A = I - \gamma P^\pi$ es invertible para todo $\gamma \in [0, 1)$. Para una secuencia $\gamma \in \{0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99\}$, calcule el número de condición mediante `kappa(A)` y grafique $\log_{10}\kappa$ versus $\gamma$. Comente la implicancia numérica de utilizar valores $\gamma$ cercanos a uno.

### Ejercicio 2 (computacional)

Implemente el algoritmo iterativo de optimalidad (*Value Iteration*) sobre el GridWorld:

$$
V_{k+1}(s) = \max_a\left[R(s, a) + \gamma \sum_{s'} P(s'|s,a) V_k(s')\right].
$$

Compare la solución $V^*$ obtenida con la $V^\pi$ bajo política uniforme. Extraiga la política óptima por $\pi^*(s) = \arg\max_a Q^*(s, a)$ y visualícela con flechas sobre la grilla. Sugerencia: extienda `dibujar_gridworld` con un parámetro `politica` que dibuje las flechas usando `geom_text` con los caracteres unicode `\u2191 \u2192 \u2193 \u2190`.

### Ejercicio 3 (estadístico)

Diseñe un experimento que compare $\varepsilon$-greedy con decaimiento ($\varepsilon_t = c/t$) contra $\varepsilon$-greedy con $\varepsilon$ fijo. Calibre $c$ por validación cruzada simulada ($c \in \{1, 5, 10, 50\}$). Reporte intervalos de confianza al 95% del regret final sobre 500 réplicas y discuta cuál variante domina.

### Ejercicio 4 (bayesiano avanzado)

Modifique `thompson_gaussiano` para que el modelo trate también la varianza $\sigma_r^2$ como desconocida, usando una prior conjugada Normal-Gamma inversa. Compare empíricamente con la versión actual cuando la varianza verdadera difiere significativamente de la asumida (por ejemplo, $\sigma_r$ verdadero $= 2$ pero el modelo asume $\sigma_r = 1$). Documente la degradación de desempeño.

---

## Cierre

El cuaderno cubre los nueve bloques anunciados en la presentación, replicando idiomáticamente en R lo realizado en Python. Quien desee corroborar la equivalencia numérica puede correr ambas versiones con las mismas semillas y comparar las salidas; salvo por diferencias en los generadores aleatorios subyacentes (Mersenne-Twister en R, PCG64 en NumPy), los resultados deterministas como la solución exacta de Bellman coinciden hasta el error de redondeo.

**Entrega evaluativa:** domingo 18 de mayo, 23:59 hs, mediante el aula virtual.

**Para consultas:** foro de la Clase 2 en Campus Virtual.